# 📘 Módulo 15 — Fine‑Tuning, LoRA e Instrução de Modelos

Este módulo mostra como adaptar modelos pré‑treinados (como BERT, GPT, T5)
para tarefas específicas usando:

- **Fine‑Tuning clássico**
- **LoRA (Low‑Rank Adaptation)**
- **Instrução de modelos (Instruction Tuning)**
- **SFT (Supervised Fine‑Tuning)**
- **RLHF (Reinforcement Learning from Human Feedback)** — visão geral

Você aprenderá:
- como ajustar modelos grandes com poucos dados
- como treinar modelos em GPUs modestas
- como criar modelos especializados
- como funciona o pipeline moderno de treinamento de LLMs

---

# 🎯 Objetivos do Módulo 15

Ao final deste módulo, você será capaz de:

✔️ entender o que é fine‑tuning e quando usar  
✔️ aplicar LoRA para treinar modelos grandes com baixo custo  
✔️ treinar um modelo para seguir instruções  
✔️ entender o pipeline SFT → RLHF  
✔️ criar seu próprio mini‑modelo instruído  

---

# 📚 Índice

1. [O que é Fine‑Tuning? Por que precisamos dele?](#oque)
2. [Fine‑Tuning Clássico com Transformers](#classico)
3. [LoRA — Low‑Rank Adaptation](#lora)
4. [Treinando um modelo com LoRA](#treino_lora)
5. [Instruction Tuning — modelos que seguem instruções](#instrucoes)
6. [SFT — Supervised Fine‑Tuning](#sft)
7. [RLHF — visão geral do processo usado em ChatGPT](#rlhf)
8. [Projeto Final — Criando seu próprio Mini‑Instruct‑GPT](#projeto)

---

# 0. Setup — bibliotecas

Vamos usar HuggingFace Transformers + PEFT (para LoRA).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

tf.__version__

<a id="oque"></a>
# 2. O que é Fine‑Tuning? Por que precisamos dele?

Modelos como BERT, GPT e T5 são **pré‑treinados** em bilhões de palavras.
Eles aprendem:
- gramática
- semântica
- conhecimento geral
- padrões de linguagem

MAS…

Eles **não** sabem automaticamente:
- classificar sentimentos
- responder perguntas específicas
- gerar textos técnicos
- seguir instruções humanas
- resolver tarefas de domínio (médico, jurídico, financeiro)

Para isso, precisamos **adaptar** o modelo para uma tarefa específica.

Esse processo é chamado de:

# 👉 Fine‑Tuning

---
# 2.1 Pré‑treino vs Fine‑Tuning

## 🧠 Pré‑treino (pretraining)
- objetivo: aprender linguagem geral
- dados: gigantescos (100GB – 1TB)
- custo: altíssimo
- feito uma vez

## 🎯 Fine‑Tuning
- objetivo: especializar o modelo
- dados: pequenos (100 – 50.000 exemplos)
- custo: baixo
- feito para cada tarefa

Exemplo:
- GPT pré‑treinado → sabe linguagem
- GPT fine‑tuned → sabe responder perguntas médicas

---
# 2.2 Por que Fine‑Tuning funciona tão bem?

Porque o modelo já aprendeu:
- sintaxe
- semântica
- relações entre palavras
- conhecimento geral

Então, com poucos exemplos, ele aprende:
- a **formulação da tarefa**
- o **estilo da resposta**
- o **formato desejado**

Isso é chamado de:

# 👉 Transfer Learning

---
# 2.3 Tipos de Fine‑Tuning

Existem três formas principais:

## 1) **Fine‑Tuning Clássico**
- atualiza **todos** os pesos do modelo
- mais preciso
- mais caro
- exige GPU forte

## 2) **Fine‑Tuning Parcial**
- congela parte do modelo
- treina só algumas camadas
- mais leve

## 3) **LoRA (Low‑Rank Adaptation)**
- adiciona camadas pequenas (adapters)
- treina só essas camadas
- modelo original fica congelado
- extremamente leve
- funciona até em GPUs de 8GB

LoRA é o padrão atual para treinar LLMs.

---
# 2.4 Quando usar Fine‑Tuning?

Use Fine‑Tuning quando você precisa:

✔️ adaptar o modelo a um domínio específico  
✔️ melhorar desempenho em uma tarefa clara  
✔️ controlar o estilo da resposta  
✔️ criar um modelo especializado  
✔️ treinar um modelo que siga instruções  

Exemplos reais:
- modelo jurídico
- modelo médico
- modelo para atendimento ao cliente
- modelo para análise de contratos
- modelo para geração de código em uma linguagem específica

---
# 2.5 Quando NÃO usar Fine‑Tuning?

❌ quando o modelo já faz a tarefa com *prompt engineering*  
❌ quando você tem poucos exemplos (< 20)  
❌ quando a tarefa é muito ampla  
❌ quando você quer apenas melhorar estilo  

Nesses casos, use:
- prompt engineering
- few‑shot prompting
- RAG (Retrieval‑Augmented Generation)

---
# 2.6 Visualização: Pré‑treino → Fine‑Tuning

import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "Pré‑treino\n(linguagem geral)", fontsize=14)
plt.text(4.1, 0.5, "Fine‑Tuning\n(tarefa específica)", fontsize=14)
plt.axis("off")
plt.title("Fluxo de treinamento de modelos modernos")
plt.show()

---
# 2.7 Por que Fine‑Tuning virou padrão?

✔️ barato  
✔️ rápido  
✔️ funciona com poucos dados  
✔️ cria modelos altamente especializados  
✔️ permite personalização profunda  

E com LoRA, ficou ainda melhor:
- treina em GPU pequena  
- não altera o modelo base  
- permite múltiplas versões (adapters)  
- combina adapters facilmente  

LoRA democratizou o treinamento de LLMs.

---
# 2.8 Conclusão desta parte

✔️ Entendemos o que é Fine‑Tuning  
✔️ Vimos por que ele é necessário  
✔️ Entendemos pré‑treino vs fine‑tuning  
✔️ Vimos quando usar e quando não usar  
✔️ Introduzimos LoRA  

Agora estamos prontos para:

**PARTE 3 — Fine‑Tuning Clássico com Transformers.**

<a id="classico"></a>
# 3. Fine‑Tuning Clássico com Transformers

Nesta parte, vamos fazer Fine‑Tuning clássico de um modelo Transformer
para uma tarefa real:

**Classificação de sentimentos (positivo/negativo)**  

Usaremos:
- modelo pré‑treinado: *distilbert-base-uncased*
- dataset: IMDb (pequeno subset)
- técnica: atualizar TODOS os pesos do modelo

Este é o método tradicional de fine‑tuning.

---
# 3.1 Instalando HuggingFace Transformers e Datasets

In [ ]:
!pip install transformers datasets --quiet

---
# 3.2 Carregando o modelo pré‑treinado (DistilBERT)

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

---
# 3.3 Carregando um dataset pequeno para treino

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:2000]")  # subset pequeno
dataset_test = load_dataset("imdb", split="test[:1000]")

dataset[0]

---
# 3.4 Tokenização do dataset

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset_tok = dataset.map(tokenize, batched=True)
dataset_test_tok = dataset_test.map(tokenize, batched=True)

dataset_tok = dataset_tok.remove_columns(["text"])
dataset_test_tok = dataset_test_tok.remove_columns(["text"])

dataset_tok.set_format("tensorflow")
dataset_test_tok.set_format("tensorflow")

---
# 3.5 Criando batches para treino

In [ ]:
train_features = {
    x: dataset_tok[x].to_tensor(default_value=0)
    for x in ["input_ids", "attention_mask"]
}
train_labels = dataset_tok["label"].to_tensor()

test_features = {
    x: dataset_test_tok[x].to_tensor(default_value=0)
    for x in ["input_ids", "attention_mask"]
}
test_labels = dataset_test_tok["label"].to_tensor()

---
# 3.6 Compilando o modelo para Fine‑Tuning

In [ ]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

---
# 3.7 Treinando o modelo (Fine‑Tuning Clássico)

In [ ]:
history = model.fit(
    train_features,
    train_labels,
    validation_split=0.1,
    epochs=2,
    batch_size=16
)

---
# 3.8 Avaliação no conjunto de teste

In [ ]:
model.evaluate(test_features, test_labels)

---
# 3.9 Fazendo previsões

In [ ]:
def predict_sentiment(text):
    tokens = tokenizer(text, return_tensors="tf", padding=True, truncation=True)
    logits = model(tokens)[0]
    pred = tf.argmax(logits, axis=1).numpy()[0]
    return "positivo" if pred == 1 else "negativo"

predict_sentiment("This movie was absolutely fantastic!")

---
# 3.10 Vantagens e desvantagens do Fine‑Tuning clássico

## ✔️ Vantagens
- máxima precisão
- modelo totalmente adaptado
- funciona muito bem com datasets grandes

## ❌ Desvantagens
- exige GPU forte
- consome muita memória
- lento
- difícil de treinar modelos grandes (7B, 13B, 70B)

Por isso, hoje o padrão é:

# 👉 LoRA (Low‑Rank Adaptation)

Ele permite treinar modelos gigantes em GPUs pequenas.

---
# 3.11 Conclusão desta parte

✔️ Carregamos um modelo pré‑treinado  
✔️ Tokenizamos um dataset real  
✔️ Fizemos Fine‑Tuning clássico  
✔️ Avaliamos e fizemos previsões  
✔️ Entendemos limitações do método tradicional  

Agora estamos prontos para:

**PARTE 4 — LoRA: Fine‑Tuning leve para modelos grandes.**

<a id="lora"></a>
# 4. LoRA — Low‑Rank Adaptation

LoRA é uma técnica criada em 2021 que revolucionou o fine‑tuning de modelos grandes.

Problema:
- modelos como LLaMA‑7B têm **bilhões** de parâmetros
- atualizar todos os pesos é caro e lento
- exige GPUs enormes

Solução:
- **não atualizar os pesos originais**
- **adicionar pequenas matrizes treináveis**
- **treinar só essas matrizes**

Isso reduz o custo em até **100×**.

---
# 4.1 A ideia central do LoRA

Em vez de atualizar uma matriz gigante W (por exemplo, 4096×4096),
LoRA adiciona duas matrizes pequenas:

- A (baixa dimensão)
- B (baixa dimensão)

E aprende apenas:

```
ΔW = B @ A
```

A matriz original W fica **congelada**.

Isso reduz drasticamente:
- memória
- custo computacional
- tempo de treino

---
# 4.2 Visualização do LoRA

import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "W (gigante)\ncongelada", fontsize=14)
plt.text(4.1, 0.5, "A (pequena)\nB (pequena)", fontsize=14)
plt.text(7.1, 0.5, "ΔW = B @ A\n(treinável)", fontsize=14)
plt.axis("off")
plt.title("Ideia central do LoRA")
plt.show()

---
# 4.3 Por que LoRA funciona tão bem?

✔️ A maior parte do conhecimento já está no modelo pré‑treinado  
✔️ Só precisamos ajustar **direções específicas**  
✔️ Essas direções têm **baixa dimensionalidade**  
✔️ LoRA aprende apenas o necessário  

Resultado:
- desempenho quase igual ao fine‑tuning completo  
- custo muito menor  
- pode treinar modelos gigantes em GPUs pequenas  

---
# 4.4 LoRA na prática (com HuggingFace PEFT)

Vamos preparar o ambiente.

In [ ]:
!pip install transformers datasets peft accelerate --quiet

---
# 4.5 Carregando um modelo base (DistilBERT)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

---
# 4.6 Aplicando LoRA ao modelo

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,                 # rank (dimensão baixa)
    lora_alpha=16,       # escala
    lora_dropout=0.1,    # dropout
    bias="none",
    task_type="SEQ_CLS"  # classificação
)

model_lora = get_peft_model(model, config)
model_lora.print_trainable_parameters()

🟣 **Interpretação**

Você verá algo como:

```
trainable params: 0.1% do total
```

Ou seja:
- 99.9% do modelo fica congelado  
- só 0.1% é treinado  

Isso é LoRA.

---
# 4.7 Carregando um dataset pequeno

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:2000]")
dataset_test = load_dataset("imdb", split="test[:1000]")

---
# 4.8 Tokenização

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset_tok = dataset.map(tokenize, batched=True)
dataset_test_tok = dataset_test.map(tokenize, batched=True)

dataset_tok = dataset_tok.remove_columns(["text"])
dataset_test_tok = dataset_test_tok.remove_columns(["text"])

dataset_tok.set_format("torch")
dataset_test_tok.set_format("torch")

---
# 4.9 Treinando com LoRA

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./lora-model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=20,
    save_strategy="no"
)

trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=dataset_tok,
    eval_dataset=dataset_test_tok
)

trainer.train()

---
# 4.10 Avaliação

In [ ]:
trainer.evaluate()

---
# 4.11 Por que LoRA é tão importante hoje?

✔️ permite treinar modelos gigantes em GPUs pequenas  
✔️ permite criar múltiplas versões do mesmo modelo  
✔️ adapters podem ser combinados  
✔️ custo baixíssimo  
✔️ desempenho excelente  

LoRA é o padrão atual para:
- LLaMA  
- Mistral  
- Falcon  
- Gemma  
- Qwen  

E praticamente todos os modelos open‑source.

---
# 4.12 Conclusão desta parte

✔️ Entendemos o que é LoRA  
✔️ Vimos por que ela funciona  
✔️ Aplicamos LoRA a um Transformer real  
✔️ Treinamos um modelo com custo reduzido  
✔️ Avaliamos e entendemos o ganho  

Agora estamos prontos para:

**PARTE 5 — Instruction Tuning: ensinando modelos a seguir instruções.**

<a id="instrucoes"></a>
# 5. Instruction Tuning — ensinando modelos a seguir instruções

Até agora, os modelos que treinamos fazem:
- classificação
- previsão de tokens
- tarefas específicas

Mas eles **não sabem seguir instruções** como:

- "Explique isso como se eu tivesse 10 anos"
- "Resuma este texto"
- "Traduza para espanhol"
- "Liste 5 ideias criativas"

Para isso, precisamos de:

# 👉 Instruction Tuning

---
# 5.1 O que é Instruction Tuning?

É o processo de treinar um modelo para:

- entender instruções humanas
- responder de forma útil
- seguir formatos pedidos
- adaptar o estilo da resposta

Usamos pares:

```
INSTRUÇÃO → RESPOSTA
```

Exemplo:

```
"Explique o que é fotossíntese"
→
"Fotossíntese é o processo pelo qual plantas produzem energia..."
```

---
# 5.2 Por que isso funciona tão bem?

Porque modelos grandes já entendem linguagem.

O Instruction Tuning ensina:
- como **interpretar pedidos**
- como **estruturar respostas**
- como **ser útil**

É como ensinar boas práticas de comunicação.

---
# 5.3 Dataset típico de Instruction Tuning

Um dataset simples pode ser assim:

instruction_data = [
    {"instruction": "Explique o que é um buraco negro.",
     "response": "Um buraco negro é uma região do espaço com gravidade tão intensa que nada escapa."},

    {"instruction": "Liste três frutas vermelhas.",
     "response": "Morango, cereja e framboesa."},

    {"instruction": "Traduza para inglês: 'O gato está no telhado'.",
     "response": "The cat is on the roof."}
]

instruction_data

---
# 5.4 Transformando instruções em formato de treino

O modelo recebe:

```
<INSTRUÇÃO>
```

E deve gerar:

```
<RESPOSTA>
```

In [ ]:
texts = []
labels = []

for item in instruction_data:
    texts.append(item["instruction"])
    labels.append(item["response"])

texts, labels

---
# 5.5 Tokenização com T5 (modelo ideal para instruções)

In [ ]:
!pip install transformers --quiet

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 5.6 Preparando dados para treino

In [ ]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="tf")
targets = tokenizer(labels, padding=True, truncation=True, return_tensors="tf")

---
# 5.7 Compilando o modelo para Instruction Tuning

In [ ]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss=model.compute_loss
)

---
# 5.8 Treinando o modelo

In [ ]:
history = model.fit(
    {"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"]},
    targets["input_ids"],
    epochs=20,
    verbose=0
)

---
# 5.9 Testando o modelo instruído

In [ ]:
def instruct(prompt):
    tokens = tokenizer(prompt, return_tensors="tf")
    output = model.generate(**tokens, max_length=50)
    return tokenizer.decode(output[0], skip_special_tokens=True)

instruct("Liste três animais que vivem no oceano.")

---
# 5.10 O que aprendemos aqui?

✔️ Como preparar dados de instrução  
✔️ Como treinar um modelo para seguir instruções  
✔️ Como testar o modelo instruído  
✔️ Como funciona o pipeline básico do Instruction Tuning  

Isso é exatamente o que deu origem ao:
- FLAN‑T5  
- Alpaca  
- LLaMA‑Instruct  
- Mistral‑Instruct  

E é a base do comportamento de assistentes como ChatGPT.

---
# 5.11 Conclusão desta parte

✔️ Entendemos o que é Instruction Tuning  
✔️ Criamos um dataset simples de instruções  
✔️ Treinamos um modelo T5 para seguir instruções  
✔️ Testamos o modelo  

Agora estamos prontos para:

**PARTE 6 — SFT: Supervised Fine‑Tuning (o passo 1 do pipeline ChatGPT).**

<a id="sft"></a>
# 6. SFT — Supervised Fine‑Tuning

O SFT é o processo de treinar um modelo para responder como um assistente humano.

Ele usa pares:

```
<INSTRUÇÃO>
<RESPOSTA IDEAL>
```

A diferença para o Instruction Tuning simples é:

- datasets maiores
- respostas mais elaboradas
- estilo de assistente
- foco em utilidade, clareza e segurança

Exemplos famosos:
- OpenAI SFT dataset (não público)
- Anthropic HH dataset
- Stanford Alpaca
- Dolly dataset
- OASST (OpenAssistant)

---
# 6.1 Exemplo de dataset SFT

sft_data = [
    {
        "instruction": "Explique a diferença entre aprendizado supervisionado e não supervisionado.",
        "response": (
            "No aprendizado supervisionado, o modelo aprende a partir de exemplos rotulados, "
            "enquanto no aprendizado não supervisionado ele tenta descobrir padrões sem rótulos."
        )
    },
    {
        "instruction": "Dê três dicas para melhorar a produtividade no trabalho.",
        "response": (
            "1. Priorize tarefas importantes.\n"
            "2. Use blocos de tempo focado.\n"
            "3. Reduza distrações digitais."
        )
    },
    {
        "instruction": "Resuma o conceito de redes neurais em uma frase.",
        "response": "Redes neurais são modelos computacionais inspirados no cérebro que aprendem padrões a partir de dados."
    }
]

sft_data

---
# 6.2 Preparando o dataset para treino

Vamos transformar cada par em um único texto:

```
<INSTRUÇÃO>
<RESPOSTA IDEAL>
```

In [ ]:
texts = []
labels = []

for item in sft_data:
    texts.append(item["instruction"])
    labels.append(item["response"])

texts, labels

---
# 6.3 Usando T5 para SFT (modelo ideal para seq2seq)

In [ ]:
!pip install transformers --quiet

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 6.4 Tokenizando instruções e respostas

In [ ]:
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="tf")
targets = tokenizer(labels, padding=True, truncation=True, return_tensors="tf")

---
# 6.5 Compilando o modelo para SFT

In [ ]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss=model.compute_loss
)

---
# 6.6 Treinando o modelo (SFT)

In [ ]:
history = model.fit(
    {"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"]},
    targets["input_ids"],
    epochs=30,
    verbose=0
)

---
# 6.7 Testando o modelo SFT

In [ ]:
def ask(prompt):
    tokens = tokenizer(prompt, return_tensors="tf")
    output = model.generate(**tokens, max_length=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

ask("Explique o que é overfitting em machine learning.")

---
# 6.8 O que o SFT faz de diferente?

✔️ Ensina o modelo a responder como um assistente  
✔️ Melhora clareza, estrutura e utilidade  
✔️ Reduz respostas vagas ou incompletas  
✔️ Prepara o modelo para RLHF  

O SFT é o **primeiro passo** do pipeline ChatGPT.

---
# 6.9 Pipeline completo (visão geral)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "1. SFT\n(respostas ideais)", fontsize=14)
plt.text(4.1, 0.5, "2. RLHF\n(preferências humanas)", fontsize=14)
plt.text(7.1, 0.5, "3. Pós‑treino\n(refinamento)", fontsize=14)
plt.axis("off")
plt.title("Pipeline moderno de treinamento de LLMs")
plt.show()

---
# 6.10 Conclusão desta parte

✔️ Entendemos o que é SFT  
✔️ Criamos um dataset simples  
✔️ Treinamos um modelo T5 com SFT  
✔️ Testamos o modelo  
✔️ Entendemos o pipeline ChatGPT  

Agora estamos prontos para:

**PARTE 7 — RLHF: Reinforcement Learning from Human Feedback (visão geral).**

<a id="rlhf"></a>
# 7. RLHF — Reinforcement Learning from Human Feedback

O RLHF é o processo que ensina o modelo a **preferir respostas humanas**.

Ele é o passo que transforma:

- um modelo instruído (SFT)
- em um **assistente alinhado ao comportamento humano**

O pipeline completo é:

1. **SFT** — modelo aprende a responder bem  
2. **Reward Model (RM)** — modelo aprende o que humanos preferem  
3. **RLHF** — modelo é otimizado para maximizar preferências humanas  

Vamos entender cada etapa.

---
# 7.1 Etapa 1 — SFT (Supervised Fine‑Tuning)

Você já fez isso na parte anterior.

O modelo aprende:
- como responder
- como estruturar respostas
- como ser útil

Mas ainda não sabe:
- qual resposta é **melhor**
- qual resposta é **mais segura**
- qual resposta é **mais alinhada**

Para isso, precisamos do próximo passo.

---
# 7.2 Etapa 2 — Reward Model (RM)

O Reward Model é treinado com **comparações humanas**.

Exemplo:

```
Pergunta: "Explique buracos negros."

Resposta A: "Buracos negros são regiões do espaço..."
Resposta B: "Buracos negros são tipo um aspirador gigante..."

Humano escolhe: A
```

O RM aprende:
- qual resposta é melhor
- qual resposta é mais clara
- qual resposta é mais segura

O RM é um modelo que recebe:

```
(instrução, resposta)
```

E retorna:

```
score de qualidade
```

---
# 7.3 Etapa 3 — RLHF (PPO)

Agora usamos **Reinforcement Learning** para ajustar o modelo.

O modelo gera respostas.
O Reward Model avalia.
O modelo é atualizado para maximizar o score.

O algoritmo mais usado:

- **PPO (Proximal Policy Optimization)**  

Ele ajusta o modelo para:
- ser mais útil
- ser mais educado
- evitar respostas perigosas
- seguir preferências humanas

---
# 7.4 Visualização do pipeline RLHF

import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.text(0.1, 0.5, "1. SFT\n(respostas ideais)", fontsize=14)
plt.text(3.6, 0.5, "2. Reward Model\n(preferências humanas)", fontsize=14)
plt.text(7.6, 0.5, "3. RLHF\n(otimização via PPO)", fontsize=14)
plt.axis("off")
plt.title("Pipeline RLHF usado em ChatGPT")
plt.show()

---
# 7.5 Exemplo simplificado de Reward Model

Aqui criamos um RM fictício só para ilustrar.

def reward_model(response):
    score = 0
    if "explica" in response.lower():
        score += 1
    if len(response.split()) > 5:
        score += 1
    return score

reward_model("Explique o que é uma estrela.")

---
# 7.6 Exemplo simplificado de RLHF

Aqui simulamos o processo:
- o modelo gera respostas
- o RM avalia
- escolhemos a melhor

responses = [
    "Uma estrela é uma bola de gás quente.",
    "Estrela é tipo uma lâmpada gigante no céu.",
    "Explique: estrela é uma esfera de plasma que brilha por fusão nuclear."
]

scores = [reward_model(r) for r in responses]
best = responses[scores.index(max(scores))]

best

---
# 7.7 O que RLHF NÃO faz

❌ não ensina fatos  
❌ não melhora conhecimento  
❌ não aumenta inteligência  

RLHF apenas:
- ajusta comportamento  
- melhora estilo  
- evita respostas ruins  
- alinha o modelo com preferências humanas  

---
# 7.8 Por que RLHF é tão importante?

✔️ torna o modelo mais seguro  
✔️ melhora a utilidade  
✔️ reduz toxicidade  
✔️ melhora clareza  
✔️ cria comportamento de assistente  

Sem RLHF, modelos seriam:
- mais secos  
- menos úteis  
- mais propensos a erros de estilo  
- menos alinhados com expectativas humanas

---
# 7.9 Conclusão desta parte

✔️ Entendemos o pipeline RLHF  
✔️ Vimos como funciona o Reward Model  
✔️ Vimos como o PPO ajusta o modelo  
✔️ Entendemos por que RLHF é essencial  

Agora estamos prontos para:

**PARTE 8 — Projeto Final: Criando seu próprio Mini‑Instruct‑GPT.**

<a id="projeto"></a>
# 8. Projeto Final — Criando seu próprio Mini‑Instruct‑GPT

Neste projeto você vai:

✔️ Criar um dataset de instruções  
✔️ Aplicar LoRA a um modelo GPT‑like  
✔️ Fazer SFT (Supervised Fine‑Tuning)  
✔️ Testar o modelo como um assistente  

Vamos usar:
- modelo base: **GPT‑2 small**
- técnica: **LoRA**
- tarefa: **seguir instruções humanas**

---
# 8.1 Instalando dependências

In [ ]:
!pip install transformers datasets peft accelerate --quiet

---
# 8.2 Criando um dataset simples de instruções

instruction_data = [
    {
        "instruction": "Explique o que é aprendizado de máquina.",
        "response": "Aprendizado de máquina é a área da computação que cria modelos capazes de aprender padrões a partir de dados."
    },
    {
        "instruction": "Liste três vantagens de usar redes neurais.",
        "response": "Elas aprendem padrões complexos, generalizam bem e funcionam em muitos tipos de dados."
    },
    {
        "instruction": "Resuma o conceito de overfitting.",
        "response": "Overfitting ocorre quando o modelo aprende demais os dados de treino e não generaliza para novos dados."
    },
    {
        "instruction": "Dê um exemplo simples de regressão linear.",
        "response": "Prever o preço de uma casa com base no tamanho em metros quadrados."
    }
]

instruction_data

---
# 8.3 Transformando instruções em formato de treino

O modelo recebe:

```
<INSTRUÇÃO>
```

E deve gerar:

```
<RESPOSTA>
```

In [ ]:
texts = []
labels = []

for item in instruction_data:
    texts.append(item["instruction"])
    labels.append(item["response"])

texts, labels

---
# 8.4 Carregando GPT‑2 como modelo base

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

---
# 8.5 Aplicando LoRA ao GPT‑2

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model_lora = get_peft_model(model, config)
model_lora.print_trainable_parameters()

---
# 8.6 Preparando dataset para treino

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "instruction": texts,
    "response": labels
})

def format_example(example):
    prompt = f"Instrução: {example['instruction']}\nResposta:"
    target = example["response"]
    return {
        "input_text": prompt,
        "target_text": target
    }

dataset = dataset.map(format_example)

def tokenize(batch):
    tokens = tokenizer(
        batch["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    labels = tokenizer(
        batch["target_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )["input_ids"]
    tokens["labels"] = labels
    return tokens

dataset_tok = dataset.map(tokenize, batched=True)
dataset_tok.set_format("torch")

---
# 8.7 Treinando o Mini‑Instruct‑GPT com LoRA

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./mini-instruct-gpt",
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=50,
    logging_steps=5,
    save_strategy="no"
)

trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=dataset_tok
)

trainer.train()

---
# 8.8 Função para gerar respostas do Mini‑Instruct‑GPT

In [ ]:
def instruct(prompt):
    text = f"Instrução: {prompt}\nResposta:"
    tokens = tokenizer(text, return_tensors="pt")
    output = model_lora.generate(
        **tokens,
        max_length=80,
        temperature=0.7,
        do_sample=True
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 8.9 Testando o modelo

In [ ]:
instruct("Explique o que é uma rede neural.")

In [ ]:
instruct("Liste três aplicações de visão computacional.")

In [ ]:
instruct("Resuma o conceito de gradiente descendente.")

---
# 8.10 Conclusões do Projeto Final

✔️ Criamos um dataset de instruções  
✔️ Aplicamos LoRA a um modelo GPT‑like  
✔️ Fizemos SFT (Supervised Fine‑Tuning)  
✔️ Treinamos um Mini‑Instruct‑GPT  
✔️ Testamos o modelo como assistente  

Você agora entende:
- como modelos como ChatGPT são treinados  
- como funciona LoRA  
- como funciona SFT  
- como funciona Instruction Tuning  
- como criar seu próprio assistente especializado  

Parabéns — você concluiu o **Módulo 15**.

O próximo passo é o **Módulo 16 — RAG, Memória e Sistemas de IA Híbridos**.